<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/cnn_fashion_mnist/cnn_fashion_mnist/cnn_fashion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN on Fashion-MNIST

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## Data

Download Fashion-MNIST and wrap it in data loaders. Images are normalized to [-1, 1].

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train = DataLoader(datasets.FashionMNIST('.', train=True,  download=True, transform=transform), batch_size=64, shuffle=True)
test  = DataLoader(datasets.FashionMNIST('.', train=False, download=True, transform=transform), batch_size=256)

## Data preview

A sample of items from the training set.

In [ ]:
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

images, labels = next(iter(train))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(class_names[labels[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## Model

Two conv blocks (conv → ReLU → pool) followed by two fully connected layers. The output has 10 logits, one per class.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)

## Training

Adam optimizer with cross-entropy loss. After each epoch, accuracy is computed on the full test set.

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for x, y in train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            predicted_classes = logits.argmax(dim=1)
            for pred, label in zip(predicted_classes, y):
                if pred == label:
                    correct += 1
                total += 1

    print(f"Epoch {epoch+1}  acc={correct / total:.3f}")

## Predictions

Run the trained model on a batch of test items. Green = correct, red = wrong. True label shown in parentheses.

In [ ]:
model.eval()
images, labels = next(iter(test))

with torch.no_grad():
    logits = model(images[:16].to(device))
    predictions = logits.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    predicted = class_names[predictions[i]]
    true      = class_names[labels[i]]
    color = 'green' if predicted == true else 'red'
    ax.set_title(f"{predicted}\n({true})", color=color, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()